# Sri Lanka History Dataset Collector — Automated, Copyright-Safe**Fully automated pipeline.** No manual file uploads. Every source fetched here iseither **public domain** (pre-1928 chronicle translations on Internet Archive /sacred-texts.com) or **openly-licensed structured data** (Wikidata, OpenStreetMap).**Sources used:**1. **Chronicles (public domain translations only)** — Wilhelm Geiger's English   translation of the *Mahavamsa* (1912) and *Culavamsa* (1929 vol. 1 is PD; check   vol. 2 date), and Robert C. Childers / Upham's 19th-century translation of the   *Rajavaliya* — fetched from Internet Archive's full-text search, not hardcoded.2. **Wikidata** (CC0) — structured facts about Sri Lankan monarchs, kingdoms, and   UNESCO World Heritage Sites, via SPARQL.3. **OpenStreetMap** (ODbL, attribution required) — coordinates for historic sites.**Deliberately NOT included** (to stay copyright/ToS-safe):- Modern historians' books still in copyright (Paranavitana, de Silva, Siriweera,  Gunawardana, Seneviratna, etc.) — you would need to legally purchase/access these  yourself and add them as a manual-upload step if you want them included.- JSTOR / Academia.edu / ResearchGate — paywalled/ToS-restricted, not scraped here.- Pujavaliya, Nikaya Sangrahaya, Dipavamsa — included only if a public-domain English  translation is found via the Internet Archive search step below; otherwise skipped  automatically and logged.**GPU:** Enable GPU (Settings → Accelerator → T4 x2) before running — needed for thelocal LLM extraction step.

## 1. Setup — install dependencies

In [ ]:
!pip install -q -U requests beautifulsoup4 SPARQLWrapper transformers accelerate bitsandbytes sentencepiece tqdm ipywidgetsprint("Dependencies installed.")

## 2. Config & progress-tracking paths

In [ ]:
from pathlib import Pathimport json, time, re, gcOUTPUT_DIR = Path('/kaggle/working/dataset')OUTPUT_DIR.mkdir(parents=True, exist_ok=True)(OUTPUT_DIR / 'progress').mkdir(exist_ok=True)(OUTPUT_DIR / 'raw_sources').mkdir(exist_ok=True)MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # smaller alt: "Qwen/Qwen2.5-3B-Instruct"CHUNK_CHARS = 3500CHUNK_OVERLAP = 300REQUEST_DELAY = 1.5   # seconds between HTTP requests — be polite to free servicesPROGRESS_FILE = OUTPUT_DIR / 'progress' / 'progress.json'ENTITIES_FILE = OUTPUT_DIR / 'entities.json'EVENTS_FILE = OUTPUT_DIR / 'events.json'LOG_FILE = OUTPUT_DIR / 'progress' / 'log.txt'def log(msg):    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"    print(line)    with open(LOG_FILE, 'a', encoding='utf-8') as f:        f.write(line + "\n")print("Output dir:", OUTPUT_DIR)

## 3. Kingdom listUsed to tag extracted facts and to query Wikidata/OSM per kingdom capital.

In [ ]:
kingdoms = [    'Anuradhapura', 'Polonnaruwa', 'Dambadeniya', 'Yapahuwa',    'Kurunegala', 'Gampola', 'Kotte', 'Sitawaka', 'Kandy']print(f"{len(kingdoms)} kingdoms configured.")

## 4. Discover public-domain chronicle translations on Internet ArchiveSearches IA's catalog by title instead of hardcoding item identifiers (which canchange) — pulls back candidates, and you eyeball `candidates` before fetching toconfirm you're grabbing the right (public-domain) edition.

In [ ]:
import requestsfrom tqdm.auto import tqdmCHRONICLE_SEARCH_TERMS = [    "Mahavamsa Geiger translation",    "Culavamsa Geiger translation",    "Rajavaliya Upham translation",    "Dipavamsa Oldenberg translation",    "Pujavaliya Sri Lanka chronicle english",    "Nikaya Sangrahaya english translation",]def ia_search(query, rows=5):    url = "https://archive.org/advancedsearch.php"    params = {        "q": query,        "fl[]": ["identifier", "title", "year", "publicdate", "licenseurl"],        "rows": rows,        "output": "json",    }    r = requests.get(url, params=params, timeout=30)    r.raise_for_status()    return r.json().get("response", {}).get("docs", [])candidates = {}pbar = tqdm(CHRONICLE_SEARCH_TERMS, desc="Searching Internet Archive")for term in pbar:    pbar.set_postfix_str(term[:30])    try:        results = ia_search(term)    except Exception as e:        log(f"  search failed for '{term}': {e}")        results = []    candidates[term] = results    for doc in results:        print(f"  [{doc.get('identifier')}] {doc.get('title')} ({doc.get('year')})")    time.sleep(REQUEST_DELAY)print(f"\nSearched {len(CHRONICLE_SEARCH_TERMS)} terms, found candidates for "      f"{sum(1 for v in candidates.values() if v)} of them.")print("Review the candidates above. Edit CONFIRMED_IA_ITEMS in the next cell")print("with the identifiers you want to actually fetch (only pre-1929 / clearly PD editions).")

## 5. Confirm & fetch chosen chronicle texts**Manual confirmation step by design** — you pick the identifiers from the searchresults above so a mis-matched or still-in-copyright edition never gets pulled inautomatically.

In [ ]:
# ---- EDIT: paste identifiers you confirmed from the search results in Cell 4 ----CONFIRMED_IA_ITEMS = {    # "identifier-from-search": {"title": "Mahavamsa (Geiger, 1912)", "source_type": "chronicle", "priority": 1},}# -----------------------------------------------------------------------------def fetch_ia_fulltext(identifier):    """Internet Archive stores OCR full text at this predictable path for text-based items."""    url = f"https://archive.org/download/{identifier}/{identifier}_djvu.txt"    r = requests.get(url, timeout=60)    if r.status_code == 200:        return r.text    return Nonechronicle_texts = {}  # identifier -> raw textpbar = tqdm(list(CONFIRMED_IA_ITEMS.items()), desc="Fetching chronicle texts")for ident, meta in pbar:    pbar.set_postfix_str(ident[:30])    raw_path = OUTPUT_DIR / 'raw_sources' / f"{ident}.txt"    if raw_path.exists():        chronicle_texts[ident] = raw_path.read_text(encoding='utf-8', errors='ignore')        continue    text = fetch_ia_fulltext(ident)    if text:        raw_path.write_text(text, encoding='utf-8')        chronicle_texts[ident] = text    else:        log(f"  FAILED to fetch {ident} (item may not have OCR text; check the identifier)")    time.sleep(REQUEST_DELAY)print(f"\n{len(chronicle_texts)} chronicle text(s) fetched and cached in raw_sources/.")if not CONFIRMED_IA_ITEMS:    print("Nothing confirmed yet — fill in CONFIRMED_IA_ITEMS above using the Cell 4 search results, then re-run this cell.")

## 6. Optional fallback — sacred-texts.comsacred-texts.com hosts several 19th/early-20th-century Ceylon chronicle translationsas plain HTML (public domain). This generic crawler follows a chapter index page yousupply, so it works whether or not the exact URL layout matches what's below —inspect the printed links before trusting the scrape.

In [ ]:
from bs4 import BeautifulSoupfrom tqdm.auto import tqdm# ---- EDIT: add index-page URLs you've confirmed by visiting sacred-texts.com yourself ----SACRED_TEXTS_INDEX_URLS = {    # "rajavaliya": "https://www.sacred-texts.com/journals/jra/index.htm",}# -------------------------------------------------------------------------------------------def fetch_sacred_texts_chapter_links(index_url):    r = requests.get(index_url, timeout=30)    r.raise_for_status()    soup = BeautifulSoup(r.text, "html.parser")    links = []    base = index_url.rsplit("/", 1)[0]    for a in soup.find_all("a", href=True):        href = a["href"]        if href.endswith(".htm") and not href.startswith("http"):            links.append(base + "/" + href)    return linksdef fetch_sacred_texts_page_text(url):    r = requests.get(url, timeout=30)    r.raise_for_status()    soup = BeautifulSoup(r.text, "html.parser")    for tag in soup(["script", "style", "nav"]):        tag.decompose()    return soup.get_text(separator="\n", strip=True)sacred_texts_content = {}for key, index_url in SACRED_TEXTS_INDEX_URLS.items():    log(f"Crawling sacred-texts index: {key}")    try:        links = fetch_sacred_texts_chapter_links(index_url)    except Exception as e:        log(f"  failed: {e}")        continue    print(f"  found {len(links)} chapter links for '{key}' — inspect before trusting:")    for l in links[:10]:        print("   ", l)    pages_text = []    pbar = tqdm(links, desc=f"Fetching '{key}' chapters")    for link in pbar:        try:            pages_text.append(fetch_sacred_texts_page_text(link))        except Exception as e:            log(f"    page fetch failed {link}: {e}")        time.sleep(REQUEST_DELAY)    sacred_texts_content[key] = "\n\n".join(pages_text)    (OUTPUT_DIR / 'raw_sources' / f"{key}_sacredtexts.txt").write_text(        sacred_texts_content[key], encoding='utf-8'    )print(f"\n{len(sacred_texts_content)} sacred-texts.com source(s) fetched.")

## 7. Chunk fetched chronicle text for the LLM

In [ ]:
def make_chunks(text, chunk_chars=CHUNK_CHARS, overlap=CHUNK_OVERLAP):    chunks = []    pos = 0    while pos < len(text):        end = min(pos + chunk_chars, len(text))        chunks.append(text[pos:end])        if end == len(text):            break        pos = end - overlap    return chunks# Combine everything fetched into one registry: source_key -> {text, meta}combined_sources = {}for ident, meta in CONFIRMED_IA_ITEMS.items():    if ident in chronicle_texts:        combined_sources[ident] = {"text": chronicle_texts[ident], "meta": meta}for key, text in sacred_texts_content.items() if "sacred_texts_content" in dir() else []:    combined_sources[key] = {"text": text, "meta": {"title": key, "source_type": "chronicle", "priority": 1}}print(f"{len(combined_sources)} total source text(s) ready for chunking.")for k, v in combined_sources.items():    print(f"  - {k}: {len(v['text'])} chars")

## 8. Load the LLM4-bit quantized so a 7B instruct model fits on a single Kaggle T4 (16GB). Drop tothe 3B model in Cell 2 if you hit memory errors.

In [ ]:
import torchfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfiglog(f"Loading model: {MODEL_NAME}")bnb_config = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_compute_dtype=torch.float16,    bnb_4bit_use_double_quant=True,    bnb_4bit_quant_type="nf4",)tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    quantization_config=bnb_config,    device_map="auto",)model.eval()log("Model loaded.")

## 9. Extraction promptExtracts *facts* with a short paraphrased note — never verbatim text from thechronicle — and marks uncertain/legendary material as such rather than presenting itas settled history.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are a careful historical-data extraction assistant working on Sri Lankan history.You will be given a chunk of text from a named public-domain chronicle translation. Extract structured facts ONLY — do not invent anything not supported by the text.Return ONLY valid JSON (no markdown fences, no preamble) matching this schema:{  "kings": [    {      "name": "string",      "kingdom": "one of the known kingdoms or null if unclear",      "reign_start": "year or null",      "reign_end": "year or null",      "dynasty": "string or null",      "predecessor": "string or null",      "successor": "string or null",      "certainty": "confirmed | disputed | legendary",      "note": "<=15 word paraphrased summary, never a direct quote"    }  ],  "events": [    {      "title": "string",      "year": "year or approximate era, or null",      "kingdom": "string or null",      "category": "religious | military | administrative | construction | diplomatic | other",      "certainty": "confirmed | disputed | legendary",      "note": "<=15 word paraphrased summary"    }  ]}If nothing relevant is in this chunk, return {"kings": [], "events": []}.Never copy full sentences from the source text into your output — paraphrase everything."""def build_user_prompt(chunk_text, source_meta):    return f"""Source: {source_meta.get('title')} ({source_meta.get('source_type')}, priority {source_meta.get('priority')})Text chunk:\"\"\"{chunk_text}\"\"\"Extract kings and events per the schema."""print("Prompt templates ready.")

## 10. LLM call wrapper (with retry + JSON repair)

In [ ]:
def call_llm(system_prompt, user_prompt, max_new_tokens=1024, temperature=0.1):    messages = [        {"role": "system", "content": system_prompt},        {"role": "user", "content": user_prompt},    ]    inputs = tokenizer.apply_chat_template(        messages, add_generation_prompt=True, return_tensors="pt"    ).to(model.device)    with torch.no_grad():        out = model.generate(            inputs,            max_new_tokens=max_new_tokens,            temperature=temperature,            do_sample=temperature > 0,            pad_token_id=tokenizer.eos_token_id,        )    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)    return text.strip()def parse_json_safe(raw_text):    cleaned = re.sub(r"^```(json)?|```$", "", raw_text.strip(), flags=re.MULTILINE).strip()    match = re.search(r"\{.*\}", cleaned, re.DOTALL)    if not match:        return None    try:        return json.loads(match.group(0))    except json.JSONDecodeError:        return Nonedef extract_from_chunk(chunk_text, source_meta, retries=2):    user_prompt = build_user_prompt(chunk_text, source_meta)    for attempt in range(retries + 1):        raw = call_llm(EXTRACTION_SYSTEM_PROMPT, user_prompt)        parsed = parse_json_safe(raw)        if parsed is not None:            return parsed        log(f"  JSON parse failed (attempt {attempt+1}), retrying...")    log("  Giving up on this chunk after retries.")    return {"kings": [], "events": []}print("LLM call wrapper ready.")

## 11. Progress state (resumable)

In [ ]:
def load_progress():    if PROGRESS_FILE.exists():        return json.loads(PROGRESS_FILE.read_text(encoding='utf-8'))    return {"completed_chunks": [], "completed_sources": [], "started_at": time.strftime('%Y-%m-%d %H:%M:%S')}def save_progress(progress):    PROGRESS_FILE.write_text(json.dumps(progress, indent=2, ensure_ascii=False), encoding='utf-8')def chunk_key(source_key, idx):    return f"{source_key}::{idx}"progress = load_progress()log(f"Resuming: {len(progress['completed_chunks'])} chunks already done.")

## 12. Main LLM extraction loop over chronicle text

In [ ]:
from tqdm.auto import tqdmall_kings = []all_events = []if ENTITIES_FILE.exists():    all_kings = json.loads(ENTITIES_FILE.read_text(encoding='utf-8'))if EVENTS_FILE.exists():    all_events = json.loads(EVENTS_FILE.read_text(encoding='utf-8'))# Pre-count total chunks across all sources so the overall bar has a real total_source_chunks = {sk: make_chunks(entry["text"]) for sk, entry in combined_sources.items()}_total_chunks = sum(len(c) for c in _source_chunks.values())_already_done = sum(1 for sk, chs in _source_chunks.items()                    for i in range(len(chs)) if chunk_key(sk, i) in progress["completed_chunks"])overall_bar = tqdm(total=_total_chunks, initial=_already_done, desc="Overall extraction progress", unit="chunk")for source_key, entry in combined_sources.items():    meta = entry["meta"]    chunks = _source_chunks[source_key]    log(f"Processing source: {source_key} ({len(chunks)} chunks)")    source_bar = tqdm(range(len(chunks)), desc=f"  {source_key}", leave=False, unit="chunk")    for idx in source_bar:        chunk_text = chunks[idx]        key = chunk_key(source_key, idx)        if key in progress["completed_chunks"]:            continue        result = extract_from_chunk(chunk_text, meta)        for k in result.get("kings", []):            k["_source"] = source_key            k["_source_title"] = meta.get("title")            k["_source_type"] = meta.get("source_type")            k["_source_priority"] = meta.get("priority")            all_kings.append(k)        for e in result.get("events", []):            e["_source"] = source_key            e["_source_title"] = meta.get("title")            e["_source_type"] = meta.get("source_type")            e["_source_priority"] = meta.get("priority")            all_events.append(e)        progress["completed_chunks"].append(key)        ENTITIES_FILE.write_text(json.dumps(all_kings, indent=2, ensure_ascii=False), encoding='utf-8')        EVENTS_FILE.write_text(json.dumps(all_events, indent=2, ensure_ascii=False), encoding='utf-8')        save_progress(progress)        overall_bar.update(1)        source_bar.set_postfix(kings=len(all_kings), events=len(all_events))        overall_bar.set_postfix(kings=len(all_kings), events=len(all_events))    if source_key not in progress["completed_sources"]:        progress["completed_sources"].append(source_key)        save_progress(progress)    log(f"Finished {source_key}.")overall_bar.close()log(f"LLM extraction complete: {len(all_kings)} raw king entries, {len(all_events)} raw event entries.")

## 13. Wikidata — structured monarch, kingdom & UNESCO site dataCC0-licensed, already structured — no LLM needed. Cross-checks / supplements whatthe chronicle extraction found.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSONWIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"def run_sparql(query):    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT, agent="SriLankaHistoryDatasetBot/1.0")    sparql.setQuery(query)    sparql.setReturnFormat(JSON)    return sparql.query().convert()["results"]["bindings"]MONARCHS_QUERY = """SELECT ?person ?personLabel ?birth ?death ?positionLabel WHERE {  ?person wdt:P39 ?position .  ?position wdt:P279* wd:Q1523447 .   # subclass of "monarch"  ?person wdt:P27 wd:Q854 .           # country of citizenship: Sri Lanka  OPTIONAL { ?person wdt:P569 ?birth }  OPTIONAL { ?person wdt:P570 ?death }  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }}LIMIT 500"""UNESCO_QUERY = """SELECT ?site ?siteLabel ?coord ?inception WHERE {  ?site wdt:P1435 wd:Q9259 .          # heritage designation: World Heritage Site  ?site wdt:P17 wd:Q854 .             # country: Sri Lanka  OPTIONAL { ?site wdt:P625 ?coord }  OPTIONAL { ?site wdt:P571 ?inception }  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }}"""wikidata_monarchs, wikidata_unesco = [], []try:    log("Querying Wikidata: Sri Lankan monarchs...")    wikidata_monarchs = run_sparql(MONARCHS_QUERY)    log(f"  {len(wikidata_monarchs)} results")except Exception as e:    log(f"  Wikidata monarch query failed: {e}")time.sleep(REQUEST_DELAY)try:    log("Querying Wikidata: Sri Lankan UNESCO World Heritage Sites...")    wikidata_unesco = run_sparql(UNESCO_QUERY)    log(f"  {len(wikidata_unesco)} results")except Exception as e:    log(f"  Wikidata UNESCO query failed: {e}")(OUTPUT_DIR / 'wikidata_monarchs.json').write_text(    json.dumps(wikidata_monarchs, indent=2, ensure_ascii=False), encoding='utf-8')(OUTPUT_DIR / 'wikidata_unesco_sites.json').write_text(    json.dumps(wikidata_unesco, indent=2, ensure_ascii=False), encoding='utf-8')print(f"Saved {len(wikidata_monarchs)} monarch records, {len(wikidata_unesco)} UNESCO site records.")

## 14. OpenStreetMap — coordinates for kingdom capitals & historic sitesODbL-licensed via the Overpass API. Attribution is required if you publish thisdataset: **"© OpenStreetMap contributors"**.

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"def overpass_search_historic(place_name, country="Sri Lanka"):    query = f"""    [out:json][timeout:25];    area["name"="{country}"]->.searchArea;    (      node["name"~"{place_name}",i]["historic"](area.searchArea);      way["name"~"{place_name}",i]["historic"](area.searchArea);    );    out center 5;    """    r = requests.post(OVERPASS_URL, data={"data": query}, timeout=60)    r.raise_for_status()    return r.json().get("elements", [])from tqdm.auto import tqdmosm_results = {}pbar = tqdm(kingdoms, desc="Querying OpenStreetMap")for kingdom in pbar:    pbar.set_postfix_str(kingdom)    try:        elements = overpass_search_historic(kingdom)        osm_results[kingdom] = elements    except Exception as e:        log(f"  OSM query failed for {kingdom}: {e}")        osm_results[kingdom] = []    time.sleep(REQUEST_DELAY)(OUTPUT_DIR / 'osm_coordinates.json').write_text(    json.dumps(osm_results, indent=2, ensure_ascii=False), encoding='utf-8')print("Saved osm_coordinates.json — remember to credit © OpenStreetMap contributors on publish.")

## 15. Validation & mergeDeduplicates king entries from the chronicle extraction, cross-references againstWikidata monarch records where names match, and flags date inconsistencies.

In [ ]:
def norm_name(name):    return re.sub(r"\s+", " ", (name or "").strip().lower())def validate_and_merge(kings, wd_monarchs):    wd_by_name = {norm_name(m.get("personLabel", {}).get("value")): m for m in wd_monarchs}    by_name = {}    for k in kings:        key = (norm_name(k.get("name")), k.get("kingdom"))        by_name.setdefault(key, []).append(k)    merged, issues = [], []    for (name_key, kingdom), entries in by_name.items():        entries_sorted = sorted(entries, key=lambda x: (x.get("_source_priority") or 99))        best = entries_sorted[0]        sources_cited = list({e["_source_title"] for e in entries if e.get("_source_title")})        best["_corroborating_sources"] = sources_cited        best["_corroboration_count"] = len(sources_cited)        best["_in_wikidata"] = name_key in wd_by_name        try:            rs, re_ = best.get("reign_start"), best.get("reign_end")            if rs is not None and re_ is not None and str(rs).isdigit() and str(re_).isdigit():                if int(re_) < int(rs):                    issues.append(f"Date inconsistency for {best.get('name')}: reign_end < reign_start")        except Exception:            pass        merged.append(best)    return merged, issuesvalidated_kings, date_issues = validate_and_merge(all_kings, wikidata_monarchs)log(f"Merged to {len(validated_kings)} unique king entries.")log(f"{sum(1 for k in validated_kings if k['_in_wikidata'])} corroborated by Wikidata.")if date_issues:    log(f"{len(date_issues)} validation issues found:")    for i in date_issues[:20]:        log(f"  - {i}")else:    log("No date-consistency issues found.")

## 16. Save final dataset

In [ ]:
FINAL_FILE = OUTPUT_DIR / 'sri_lanka_kings_validated.json'FINAL_EVENTS_FILE = OUTPUT_DIR / 'sri_lanka_events.json'ISSUES_FILE = OUTPUT_DIR / 'validation_issues.json'FINAL_FILE.write_text(json.dumps(validated_kings, indent=2, ensure_ascii=False), encoding='utf-8')FINAL_EVENTS_FILE.write_text(json.dumps(all_events, indent=2, ensure_ascii=False), encoding='utf-8')ISSUES_FILE.write_text(json.dumps(date_issues, indent=2, ensure_ascii=False), encoding='utf-8')log(f"Saved: {FINAL_FILE}")log(f"Saved: {FINAL_EVENTS_FILE}")log(f"Saved: {ISSUES_FILE}")print("\nDone. Files in", OUTPUT_DIR, ":")for p in sorted(OUTPUT_DIR.rglob('*')):    if p.is_file():        print(" -", p.relative_to(OUTPUT_DIR))print("\nAttribution reminder for publishing this dataset:")print(" - Wikidata content: CC0")print(" - OpenStreetMap content: (c) OpenStreetMap contributors, ODbL")print(" - Chronicle texts: public-domain translations (cite translator/edition per source)")

## 17. (Optional) Free GPU memory

In [ ]:
del modelgc.collect()torch.cuda.empty_cache()log("GPU memory released.")